# Cookbook

Part 1 takes a single indicator from a row in the workbook
to a finished, validated deliverable — including the
evidence, the data licensing, the map you use to check it,
and how you hand it back.

Part 2 is a set of recipes for the other shapes your data
might take. Once you have been through Part 1, a recipe is
just a replacement for one step.

Everything runs on the demo data in `data/demo/`, so you
can execute this notebook now, before you have data of your
own.

| | |
|---|---|
| **Part 1** | One indicator, start to finish |
| **Part 2** | Recipes: points, polygons, lines, rasters, tables |
| **Part 3** | Reference: aggregation methods, error messages |

Schema version 1.0.0.

## The demo data

Real Mexicali layers, cut down so they are quick and small
enough to commit — see
[`data/demo/README.md`](../data/demo/README.md) for what
each one is and where it came from. They are here to
demonstrate the tooling; they are not a basis for indicator
values, and one column (`has_sidewalk_FABRICATED`) is
invented.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

import uli

DEMO = os.path.join('..', 'data', 'demo')
GPKG = os.path.join(DEMO, 'demo.gpkg')

# Normally your own details; this notebook is a demonstration.
ANALYST_DETAILS = {
    'name': 'Cookbook demo',
    'email': None,
    'institution': None,
}

print('uli', uli.SCHEMA_VERSION)
print('reference geographies:', uli.geography.available())

In [ ]:
import pyogrio
for name, kind in pyogrio.list_layers(GPKG):
    layer = gpd.read_file(GPKG, layer=name)
    print(f'{name:18s} {len(layer):>6,} {kind:12s} '
          f'{[c for c in layer.columns if c != "geometry"]}')

---
---
# Part 1 — One indicator, start to finish

The indicator: **#187, access to public open space**,
measured as the share of each area that is public open
space.

Eleven steps. Steps 1–4 are desk work and take longer than
the code.

## 1. Read what the team already recorded

Your work package notebook has a brief for each indicator,
and `uli.metadata_stub()` pre-fills a metadata document
from the same source. Start there rather than from a blank
page.

In [ ]:
meta = uli.metadata_stub(187, analyst=ANALYST_DETAILS)

print('indicator :', meta['indicator']['name_en'])
print('domain    :', meta['indicator']['domains'])
print('lenses    :', [m['lens'] for m in meta['measures']])
print()
print('adapted from:')
print(' ', meta['indicator']['adapted_from'][:140])
print()
print('draft rationale from the workbook:')
print(' ', meta['rationale']['statement'][:200])

The `adapted_from` reference is **provenance, not
evidence**: it records where the indicator idea came from,
not that it affects health. Supplying that link is step 2,
and it is the part of the job only you can do.

## 2. Write the causal pathway

Before searching for anything, finish this sentence:

> *[what I measure]* changes *[a mechanism]*, which changes
> *[a behaviour or exposure]*, which affects *[a health
> outcome]*.

For this indicator:

> **Public open space near where people live** provides
> somewhere to walk, exercise and meet others, and offers
> shade and cooler surfaces, which increases recreational
> physical activity and social contact and reduces heat
> exposure, which is associated with better mental health,
> lower cardiovascular risk and lower heat-related illness.

That sentence determines everything downstream: which
evidence is relevant, which threshold to use, and whether
the measure you are about to compute is the right one. If
it is hard to write, that usually means the indicator needs
discussion with the group rather than more GIS.

The mechanisms are recorded as `health_pathways`, using the
controlled list:

In [ ]:
for key, description in uli.vocab.HEALTH_PATHWAYS.items():
    print(f'{key:32s} {description}')

## 3. Find evidence for that pathway

You need at least one citation, **independent of the
article the indicator was adapted from**, showing a
meaningful health or wellbeing benefit.

Where to look: PubMed, Scopus, Google Scholar, WHO
publications, *Environment International*, *Health &
Place*, *Journal of Transport & Health*. Prefer
meta-analyses and systematic reviews, then reputable
guidance, then cohort studies.

Search the **exposure and the outcome**, not the indicator
name — for example
`"green space" AND (mortality OR "mental health") AND
(review OR meta-analysis)`.

Record what the source actually says, including the effect
size and its uncertainty where one is given. Some sources,
including narrative reviews and guidance documents, do not
report a pooled estimate; say so rather than inventing one.

> The reference below is real and is used to show the
> shape of the record. **Read your own sources and record
> what they say** — do not copy this one across.

In [ ]:
meta['rationale']['statement'] = (
    'Public open space within walking distance gives '
    'residents somewhere to walk, exercise and meet '
    'others, and provides shade and cooler surfaces in a '
    'city where summer heat limits time outdoors. Access '
    'is associated with higher recreational physical '
    'activity, more social contact and better mental '
    'health. In Mexicali the benefit is likely '
    'conditional on shade and water provision, and on the '
    'time of day the space can be used; households '
    'without air conditioning or a car are least able to '
    'substitute other options.'
)

meta['rationale']['health_pathways'] = [
    'physical_activity_recreation',
    'social_interaction',
    'heat_exposure',
    'restoration_and_mental_health',
]

meta['rationale']['arid_context'] = (
    'Most green space and health evidence comes from '
    'temperate cities. In Mexicali, summer maxima above '
    '45 C plausibly make shade, water and opening hours '
    'more binding than distance. Read alongside the WP02 '
    'thermal comfort outputs; unshaded open space may '
    'deliver much less benefit than the same area of '
    'shaded space.'
)

meta['rationale']['evidence'] = [
    {
        'claim': (
            'Access to urban green space is associated '
            'with improved mental health and wellbeing, '
            'reduced cardiovascular morbidity and '
            'increased levels of physical activity.'
        ),
        'citation': (
            'World Health Organization Regional Office '
            'for Europe (2016). Urban green spaces and '
            'health: a review of evidence. Copenhagen: '
            'WHO Regional Office for Europe.'
        ),
        'doi': None,
        'url': ('https://www.who.int/europe/publications/'
                'i/item/WHO-EURO-2016-3352-43111-60341'),
        'evidence_type': 'expert_guidance',
        'population': (
            'General urban populations, predominantly '
            'European studies.'
        ),
        'exposure': 'Access to urban green space',
        'outcome': (
            'Mental health, cardiovascular morbidity, '
            'physical activity'
        ),
        'effect': (
            'Narrative review; consistent direction of '
            'association reported across studies, no '
            'pooled effect estimate given.'
        ),
        'threshold_support': (
            'The review discusses accessibility within '
            'walking distance rather than endorsing a '
            'single distance; the project default of 500 '
            'm is used here and should be revisited '
            'against Mexicali-specific evidence.'
        ),
        'notes': (
            'Used to demonstrate the record. Supplement '
            'with a quantitative review before '
            'publication.'
        ),
    },
]

print(len(meta['rationale']['evidence']), 'evidence entry')

## 4. Find the data, and record it as you go

Record the source **when you download it**, not months
later when you have forgotten the version and the date.

Five things are required, and the validator will ask for
them: citation, URL, date retrieved, licence, and whether
the source covers Condesa. `redistributable` matters
because Reimagina Urbana cannot publish a layer whose
licence forbids it.

Put the raw file in `data/raw/<your_work_package>/` and do
not edit it by hand — every change should happen in code,
so it can be repeated. Raw data is **not** committed: it
can be large, and licences often do not allow
redistribution.

Likely sources for this project: INEGI (censo, ENIGH,
ENVI, DENUE, marco geoestadístico), the IMIP Mexicali
geovisor, municipal open data, SEMARNAT and the Mexicali
air quality network, CONAGUA, OpenStreetMap, Sentinel-2
and Landsat, GHSL, WorldPop.

In [ ]:
meta['data_sources'] = [
    {
        'name': 'Public open space, Mexicali',
        'custodian': 'OpenStreetMap contributors',
        'citation': (
            'OpenStreetMap contributors (2026). Geofabrik '
            'Mexico extract, 10 April 2026. Public open '
            'space derived using the GHSCI open space '
            'method.'
        ),
        'url': 'https://download.geofabrik.de/',
        'date_retrieved': '2026-04-10',
        'licence': 'ODbL-1.0',
        'licence_url': ('https://opendatacommons.org/'
                        'licenses/odbl/'),
        'redistributable': True,
        'spatial_resolution': 'vector polygons',
        'temporal_coverage': '2026',
        'condesa_coverage': 'full',
        'notes': (
            'Volunteered data. Completeness varies and is '
            'likely lower in recently developed areas, '
            'which matters for Condesa.'
        ),
    },
]

# What is still outstanding at this point?
for path in uli.todos(meta):
    print(path)

## 5. Compute the indicator

One call. The helper is chosen by the shape of the data —
polygons whose coverage you want, so `areal_share` (Part 2,
Recipe 3).

Compute at the finest scale your data genuinely support.
Here that is the 100 m grid, which also means the result
reaches all 40 Condesa fraccionamientos rather than the 33
a manzana-based calculation would reach.

In [ ]:
open_space = gpd.read_file(GPKG, layer='open_space')
print(f'{len(open_space)} open space polygons, '
      f'{open_space.geometry.area.sum() / 1e6:.1f} km2 total')

native = uli.areal_share(open_space, 'grid_100m',
                         as_percentage=True)
print()
print(native.head())
print()
print(native['value'].describe().round(2))

`native` is the whole output of the calculation: one row
per 100 m cell, `geo_id` and `value`. Everything from here
is bookkeeping.

## 6. Record how you did it

The method summary should let someone else reproduce the
number from the sources you documented. Write it now, while
you remember the details.

In [ ]:
meta['method']['summary'] = (
    'Public open space polygons were dissolved to remove '
    'overlaps, intersected with each 100 m grid cell, and '
    'the intersecting area divided by the cell area to '
    'give the percentage of each cell that is public open '
    'space. Cell values were aggregated to coarser '
    'reporting geographies as an area-weighted mean, '
    'since open space coverage is a property of land '
    'rather than of people.'
)
meta['method']['software'] = ['python', 'geopandas', 'uli']
meta['method']['notebook'] = 'notebooks/00b_cookbook.ipynb'
meta['method']['assumptions'] = [
    'All mapped public open space is publicly accessible '
    'in practice, which OpenStreetMap does not verify.',
]
meta['method']['limitations'] = [
    'Area of open space says nothing about its quality, '
    'shade, water provision or opening hours, which in an '
    'arid city plausibly matter more than area.',
]
meta['method']['condesa_treatment'] = (
    'Computed natively on the 100 m grid, which covers '
    'the full Condesa extent. OpenStreetMap coverage of '
    'the new development is likely incomplete, so a low '
    'value there may reflect mapping rather than absence.'
)
meta['indicator']['status'] = 'draft'
print('recorded')

## 7. Fill in the other geographies

You computed at one scale; the project reports at five.
`harmonise` produces them all from your single
calculation, using the shared crosswalk, and records how
each value got there.

`method` describes what the value *is*, which determines
how it should be combined:

| Your measure is… | Use |
|---|---|
| something people experience (access, exposure, comfort) | `population_weighted_mean` |
| a property of land (cover, temperature, land use) | `area_weighted_mean` |
| a property of streets | `length_weighted_mean` |
| a count of things | `sum` |

In [ ]:
results = uli.harmonise(
    native,
    native_scale='grid_100m',
    method='area_weighted_mean',
)

results.groupby(['geo_level', 'aggregation_method']).agg(
    units=('value', 'size'),
    with_value=('value', 'count'),
    median=('value', 'median'),
).round(2)

Note `manzana` is marked `replicated`: manzanas are
smaller than 100 m cells, so those values carry no
variation at that scale. That is recorded honestly rather
than hidden, and the index step knows to discount it.

## 8. Describe the measure and label the rows

A **measure** is one specific way of measuring the
indicator. This indicator could be measured several ways —
distance to the nearest open space, count within 500 m,
percentage cover. Here there is one, so the stub's other
lenses are dropped.

In [ ]:
measure = meta['measures'][0]
measure.update(
    id='access_to_public_open_space__density_percent_cover',
    lens='density',
    name_en='Public open space, percentage of area',
    name_es='Espacio público abierto, porcentaje del área',
    description=(
        'Percentage of the unit covered by public open '
        'space polygons.'),
    unit='percent',
    value_type='percentage',
    direction='higher_is_better',
    denominator_type='area_sqkm',
    coverage_basis='area',
    temporal_basis='point_in_time',
    native_scale='grid_100m',
    aggregation_method='area_weighted_mean',
    include_in_index=True,
    index_notes=(
        'Highly skewed: most units have no open space at '
        'all. Consider a transform or a threshold form.'),
)
meta['measures'] = [measure]

results = uli.label(results, meta, measure['id'])
results.head(3)

Those twelve columns are the delivery format, and they are
the same for every indicator in the project. `coverage`,
`quality_flag` and `native_scale` are what let someone
else judge how much to trust a given row.

## 9. Map it before you believe it

The fastest check on a spatial indicator is to look at it.

In [ ]:
grid = uli.geography.load('grid_100m').merge(
    results.query("geo_level == 'grid_100m'"),
    on='geo_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

grid.plot(column='value', ax=axes[0], legend=True,
          vmax=grid['value'].quantile(0.99),
          missing_kwds={'color': 'lightgrey'})
uli.geography.load('condesa_fraccionamiento').boundary.plot(
    ax=axes[0], color='crimson', linewidth=0.8)
axes[0].set_title('% public open space, 100 m grid '
                  '(Condesa outlined)')
axes[0].set_axis_off()

ageb = uli.geography.load('ageb').merge(
    results.query("geo_level == 'ageb'"),
    on='geo_id', how='left')
ageb.plot(column='value', ax=axes[1], legend=True,
          missing_kwds={'color': 'lightgrey'})
axes[1].set_title('the same values aggregated to AGEB')
axes[1].set_axis_off()

Questions worth asking of that map:

- Are the high values where you would expect parks to be?
- Is anywhere suspiciously blank — and is that a real
  absence, or missing data?
- Does Condesa look plausible? Here it is largely empty,
  which is consistent with a new development, but could
  equally be incomplete OpenStreetMap coverage. That
  ambiguity belongs in `method.condesa_treatment`, which is
  why step 6 said it.
- Do the two panels tell the same story? If aggregating
  changes the pattern, check the method.

## 10. Validate

`uli.check()` runs the schema, consistency and
project-requirement checks together.

In [ ]:
report = uli.check(results, meta)
print(report)

`[ERROR]` lines block delivery; `[WARN]` lines are worth
reading but will not stop you. The replication warning here
is expected and correct.

## 11. Deliver, and hand it back

`write_indicator` writes three files and refuses to write
at all if validation failed. While you are still working,
`allow_failure=True` saves a draft anyway.

In [ ]:
# This demonstration writes to a temporary folder so it
# does not leave a stray deliverable in outputs/.  Your own
# work omits output_dir, and the files land under
#   outputs/<work_package>/<indicator_code>/
import tempfile
demo_output = tempfile.mkdtemp()

written = uli.write_indicator(results, meta,
                              output_dir=demo_output)

print()
print('would normally be written to:')
print('  outputs/' + meta['indicator']['work_package']
      + '/' + meta['indicator']['code'] + '/')
print()
for key in ('results', 'metadata', 'validation'):
    path = written[key]
    print(f'{key:11s} {os.path.basename(path):55s} '
          f'{os.path.getsize(path) / 1024:>8,.0f} KB')

In [ ]:
# What the metadata document looks like on disk:
with open(written['metadata'], encoding='utf-8') as f:
    print(''.join(f.readlines()[:28]))

(This indicator belongs to WP01, so the demo reports that
path. Yours will name your own work package.)

### Handing it back

Results are gzipped, so one delivery is about 0.3 MB
rather than 5 MB, and the whole indicator set is around
26 MB — small enough to live in the repository:

```bash
git checkout -b wp04-open-space          # your package
git add outputs/WP04_greenness_and_land_cover/
git add notebooks/04_greenness_and_land_cover.ipynb
git commit -m "WP04: public open space coverage"
git push -u origin wp04-open-space
```

Then open a pull request. That gives the work a review
point and a record of what changed, which email does not.

**Commit:** the three files in `outputs/`, and your
notebook with its cells run.

**Do not commit:** anything in `data/raw/`. Source data is
often large and frequently cannot be redistributed; the
metadata records where it came from so someone else can
fetch it.

If you cannot use git, send the `outputs/<indicator>/`
folder and your notebook to the project lead, who will
commit them.

---

That is the whole process. Part 2 replaces step 5 —
everything before and after it is the same every time.

---
---
# Part 2 — Recipes

Eight ways of getting from source data to `native`, the
two-column table that step 5 produced. Find the one that
matches your data.

| Your data looks like | Recipe |
|---|---|
| Points (shops, clinics, bus stops) | 1 how many · 2 how far |
| Polygons (parks, flood zones, land use) | 3 how much cover · 4 which class |
| Lines (streets, cycle lanes) | 5 |
| A raster (NDVI, temperature, pollution) | 6 |
| A table of values per AGEB or manzana | 7 |
| One number for the whole city | 8 |

**Each helper returns a table of `geo_id` and `value`** —
one measure, one column of numbers. An indicator with
several measures (a distance *and* a count, say) means
several calls: compute each, `uli.label()` each with its
own `measure_id`, and combine them at the end with
`uli.assemble()`. Recipe 1 shows that.

---
## Recipe 1 — Points → how many, or how dense

**Use this when:** you have point locations and want a count, a count per km², or a count per 1,000 people.

**Indicators like:** access to shops, clinics, bus stops; destination counts; incident counts.

In [ ]:
destinations = gpd.read_file(GPKG, layer='destinations')
shops = destinations[destinations['category'] == 'convenience']

count = uli.count_features(shops, 'ageb')
print(count.head(3))

The `per` argument changes what is measured, so each is a
**separate measure** with its own `measure_id`, unit and
direction — not extra columns on one table.

In [ ]:
density = uli.count_features(shops, 'ageb', per='sqkm')
per_capita = uli.count_features(shops, 'ageb',
                                per='1000_persons')

# Three separate two-column tables:
for name, table in [('count', count), ('per_sqkm', density),
                    ('per_1000_persons', per_capita)]:
    print(f'{name:18s} {list(table.columns)}  '
          f'median={table["value"].median():.2f}')

# Deliver several measures for one indicator like this:
#   a = uli.label(uli.harmonise(count, 'ageb'), meta,
#                 'my_code__quantity')
#   b = uli.label(uli.harmonise(density, 'ageb'), meta,
#                 'my_code__density_per_sqkm')
#   results = uli.assemble([a, b])

**Watch out:** a count per 1,000 people is empty where nobody lives. That is deliberate — dividing by zero population would give infinity, which the schema does not allow.

---
## Recipe 2 — Points → how far away

**Use this when:** you want the distance to the closest one.

**Indicators like:** the `proximity` lens on any "access to X" indicator.

In [ ]:
markets = destinations[
    destinations['category'] == 'fresh_food_market']

native = uli.distance_to_nearest(markets, 'grid_100m', cap=3000)
print(native['value'].describe().round(0))

**Watch out:** this is **straight-line** distance, not distance along streets. Walking distance is longer, and the difference varies with the street layout. Say which you used in the measure description. WP01 uses network distance from the GHSCI pipeline.

---
## Recipe 3 — Polygons → how much of the unit they cover

**Use this when:** you want a percentage of area: tree canopy, flood extent, park cover, built-up land.

**Indicators like:** vegetation percent, flooding, residential area, public open space.

In [ ]:
open_space = gpd.read_file(GPKG, layer='open_space')

native = uli.areal_share(open_space, 'ageb', as_percentage=True)
print(native['value'].describe().round(1))

**Watch out:** overlapping polygons are dissolved first, so a park mapped twice in your source is not counted twice.

---
## Recipe 4 — Polygons → which class dominates

**Use this when:** your polygons carry a category rather than a quantity.

**Indicators like:** land use type, climate zone, hazard category.

In [ ]:
land = gpd.read_file(GPKG, layer='land_classes')
print(land['land_class'].tolist())
print()

native = uli.dominant_class(land, 'condesa_fraccionamiento',
                            'land_class')
print(native['value'].value_counts(dropna=False))

**Watch out:** the result is a **label**, not a number, so it cannot enter the index directly — deliver it as an `ordinal` measure with a documented encoding, or use Recipe 3 on one filtered class to get "percent residential", which is usually more useful. Note that 7 of the 40 fraccionamientos come back empty here: the census layer does not cover them.

---
## Recipe 5 — Lines → how much of the network has something

**Use this when:** your data is an attribute of street segments and you want the share of street length that has it.

**Indicators like:** sidewalk availability, street lighting, cycling infrastructure, traffic stress.

In [ ]:
streets = gpd.read_file(GPKG, layer='streets_condesa')
print(streets['highway'].value_counts().head())
print()

# has_sidewalk_FABRICATED is invented demo data, not a finding.
native = uli.network_share(
    streets, 'condesa_fraccionamiento',
    attribute='has_sidewalk_FABRICATED')
print(native['value'].describe().round(2))

**Watch out:** segments are cut at unit boundaries, so a road crossing three units is shared between them by length. Units with no streets return an empty value rather than zero.

---
## Recipe 6 — A raster → summarise it within each unit

**Use this when:** you have a continuous surface from satellite imagery or a model.

**Indicators like:** NDVI, land surface temperature, pollutant concentrations, urban heat.

In [ ]:
RASTER = os.path.join(DEMO, 'demo_population_100m.tif')

native = uli.zonal_statistic(RASTER, 'ageb', 'mean')
print(native['value'].describe().round(1))

**Watch out:** this reads the raster once per unit, so it is slow over the 22,355 cells of `grid_100m` — test on `ageb` first. Check the nodata value is being honoured, too: a nodata of -9999 averaged into a mean is easy to miss and hard to spot later.

---
## Recipe 7 — A table you already have per unit

**Use this when:** your values already exist for AGEBs or manzanas — from the census, or a spreadsheet from a colleague.

**Indicators like:** census variables, housing costs, employment, administrative data.

In [ ]:
census = pd.read_csv(
    os.path.join(DEMO, 'demo_ageb_census.csv'),
    dtype={'CVEGEO': str})

# Your key must match geo_id exactly.  For ageb and manzana,
# geo_id is the INEGI CVEGEO.
native = census.rename(
    columns={'CVEGEO': 'geo_id', 'pct_65_plus': 'value'}
)[['geo_id', 'value']]

known = set(uli.geography.units('ageb')['geo_id'])
print(f'{native["geo_id"].isin(known).sum()} of {len(native)} '
      'rows matched a reference AGEB')

**Watch out:** if few rows match, it is usually the key being read as a number — CVEGEO has leading zeros, so read it as text with `dtype={"CVEGEO": str}`.

---
## Recipe 8 — One number for the whole city

**Use this when:** your source only supports a single city-wide figure — a household survey, one monitoring station.

**Indicators like:** housing affordability from ENIGH/ENVI, city-level survey measures.

In [ ]:
native = pd.DataFrame({
    'geo_id': ['MX_Mexicali_2025'],
    'value': [42.0],
})

example = uli.harmonise(native, native_scale='city')
print(example.groupby(['geo_level', 'aggregation_method']).size())

**Watch out:** every finer unit gets the same value, flagged `replicated`. That is the honest representation, and the composite index excludes such measures at fine scales because a constant cannot distinguish one part of the city from another.

---
---
# Part 3 — Reference

## The steps that never change

Whichever recipe produced `native`:

```python
results = uli.harmonise(native, native_scale='grid_100m',
                        method='area_weighted_mean')
results = uli.label(results, meta, 'my_code__lens')
print(uli.check(results, meta))
uli.write_indicator(results, meta)
```

## When something goes wrong

| Message | What it means | What to do |
|---|---|---|
| `FileNotFoundError: Reference geographies not found` | `geography/` missing, or the notebook is running from another folder | run from `notebooks/`; check `uli.geography.available()` |
| `'x' is not a reporting geography` | a typo in a level name | one of `city`, `ageb`, `grid_1000m`, `condesa_fraccionamiento`, `grid_100m`, `manzana`, `condesa_lote` |
| `... is finer than ...; use method='replicated'` | you asked to aggregate *down* | let `harmonise` decide; it replicates and flags |
| `measure id ... must start with ...` | `measure_id` does not match `indicator_code` | use `<indicator_code>__<lens>` |
| `N metadata fields still contain a TODO` | the stub is not filled in | `uli.todos(meta)` lists which |
| `only N of 40 Condesa fraccionamientos have a value` | your native scale does not reach Condesa | compute on `grid_100m` if the data support it |
| Values all empty after a merge | join keys do not match | check `dtype={'CVEGEO': str}`, compare a few ids directly |
| Everything zero or absurd | often a CRS mismatch | project data is EPSG:6366; the helpers reproject inputs, so check the input file's own CRS |

If none of that fits, bring the error message and the cell
that produced it to the group.

## Where else to look

| | |
|---|---|
| Function names and arguments | [`docs/cheatsheet.md`](../docs/cheatsheet.md) |
| The reasoning behind the requirements | [`docs/analyst_guide.md`](../docs/analyst_guide.md) |
| Exact output specification | [`schema/ULI_output_schema.md`](../schema/ULI_output_schema.md) |